# S&P 500 Options: Causal DML Execution

This notebook estimates the effect of the variance-risk-premium treatment on the
return-to-expiry outcome. It declares the request through the shared causal boundary and exposes
the resolved estimand, timing, confounders, nuisance model, covariance design, and refutation
protocol before execution.

`11_model_analysis` interprets the causal estimates. This notebook validates the computation
and publishes its artifact only.

Prerequisites: `03_financial_features`, `04_model_based_features`, and `05_evaluation`.

In [1]:
"""Execute the declared S&P 500 options causal DML request."""

import polars as pl

from case_studies.research import causal_supersedes
from case_studies.sp500_options.research_workflow import open_study

In [2]:
EXECUTION_TIER = "canonical"
WORKSPACE: str = ""
PREVIEW_REDUCTIONS: dict = {}
# Retired by this run: the block-permutation refutation now compares the HAC t-statistic
# rather than the raw effect, so CAUSAL_RUNNER_VERSION moved and every causal identity with
# it. The rows named here hold a p-value computed on the shrunken placebo effects; this run
# supersedes them rather than correcting them, because the statistic is different, not the
# arithmetic. Read out of each registry's current canonical identity per label, 2026-09-10.
SUPERSEDES_CAUSAL: str = "d034b82943c5"

## Declared and resolved request

A preview must declare all sample, symbol, fold, or placebo reductions. Canonical execution uses
the complete pre-holdout analysis population.

### What `SUPERSEDES_CAUSAL` retires here

`CausalResult.one` resolves a label to exactly one canonical identity, so a refit has to name the
identity it replaces or the registry is left with two and refuses. The retired identity is
`d034b82943c5`.

What changed is the refutation statistic, not the fit. The placebo loop used to compare each
permuted run's *effect estimate* against the observed effect. Block-permuting the treatment frees
it from the controls, so the first stage can no longer predict it and its residual keeps nearly
all its variance. That residual variance is the whole denominator of the second-stage effect, so
every placebo effect is divided by a larger number than the observed one and the placebo
distribution comes out narrower than the null it stands for. The bias runs one way, toward a
refutation that reads as passed. The comparison is now on the HAC t-statistic, which carries the
denominator in it and cancels the inflation.

The retired identity fitted the same 166,105 observations and reported the same effect of 0.4098
with a HAC standard error of 0.3509, so p = 0.243 under either statistic. Its refutation p was
0.0099, the smallest value 100 draws can report, for an effect whose own t-statistic is 1.17.
Sitting at that floor is the signature. This notebook registers and hands off; the new row is
read and interpreted in `11_model_analysis`.

In [3]:
study = open_study(execution_tier=EXECUTION_TIER, workspace=WORKSPACE or None)
request_table = pl.DataFrame(
    {
        "method": ["dml"],
        "label": ["ret_to_expiry"],
        "config_name": ["dml"],
        "execution_tier": [EXECUTION_TIER],
    }
)
request_table

method,label,config_name,execution_tier
str,str,str,str
"""dml""","""ret_to_expiry""","""dml""","""canonical"""


In [4]:
request = study.causal(
    **request_table.row(0, named=True),
    preview_reductions=PREVIEW_REDUCTIONS,
    supersedes=causal_supersedes(
        study,
        SUPERSEDES_CAUSAL,
        "ret_to_expiry",
        labels=["ret_to_expiry"],
        execution_tier=EXECUTION_TIER,
    ),
)
resolved = request.resolve()
computation = resolved.spec["computation"]
estimand = computation["estimand"]
causal_plan = pl.DataFrame(
    {
        "treatment": [estimand["treatment"]],
        "outcome": [estimand["outcome"]],
        "confounders": [", ".join(estimand["confounders"])],
        "treatment_observed_at": [estimand["treatment_observed_at"]],
        "outcome_horizon": [estimand["outcome_horizon"]],
        "folds": [computation["cv"]["n_folds"]],
        "embargo_periods": [computation["cv"]["embargo_periods"]],
        "nuisance_model": [computation["model"]["class"]],
        "covariance": ["HAC with the outcome horizon"],
        "placebo_method": [computation["refutation"]["method"]],
        "placebo_block": [computation["refutation"]["block_size"]],
        "placebo_block_basis": [computation["refutation"]["block_size_basis"]],
        "analysis_rows": [computation["analysis_population"]["n_rows"]],
        "training_hash": [resolved.identity],
    }
)
causal_plan

treatment,outcome,confounders,treatment_observed_at,outcome_horizon,folds,embargo_periods,nuisance_model,covariance,placebo_method,placebo_block,placebo_block_basis,analysis_rows,training_hash
str,str,str,str,str,i64,i64,str,str,str,i64,str,i64,str
"""vrp_21d""","""ret_to_expiry""","""rv_21d, vrp_mom_5d, spread_pct…","""decision_timestamp""","""35 days 00:00:00""",5,35,"""sklearn.ensemble.HistGradientB…","""HAC with the outcome horizon""","""within_symbol_contiguous_block…",35,"""label_buffer""",206197,"""fe9f19d47fb6"""


## Execute and validate

The shared DML runner fails on missing confounders, invalid temporal folds, incomplete nuisance
fits, or a non-finite HAC standard error. A cached result must match the complete resolved
identity before it can be reused.

In [5]:
if EXECUTION_TIER == "preview" and (not WORKSPACE or not PREVIEW_REDUCTIONS):
    raise ValueError("preview execution requires WORKSPACE and PREVIEW_REDUCTIONS")
result = resolved.run()
if not result.complete or result.hash != resolved.identity:
    raise RuntimeError("causal execution did not publish the complete resolved request")

~/ml4t/public-causal-refutation/case_studies/utils/causal.py:1816: UserWarning: block permutation with block_size=35 cannot move 14.06% of the treatment rows: they sit in segments too short to hold two blocks, so the placebo distribution holds them at their observed values and the refutation p-value is biased toward 1. Read placebo_frozen_fraction alongside the p-value, and lower block_size or widen gap_tolerance if the frozen share is large.
  results = run_dml_analysis(


In [6]:
artifact = pl.DataFrame(
    {
        "causal_hash": [result.hash],
        "label": [resolved.spec["label"]],
        "execution_tier": [result.execution_tier],
        "complete": [result.complete],
    }
)
artifact

causal_hash,label,execution_tier,complete
str,str,str,bool
"""fe9f19d47fb6""","""ret_to_expiry""","""canonical""",true


The registered causal artifact is the handoff to `11_model_analysis`. No estimate or empirical
conclusion is interpreted here.